# v2 Notebook 03: Train the trees and test them four ways

**Author:** Anush Jindal

This is the long notebook (about 30 minutes on a 2-core CPU). It trains every tree of v2 and saves the scores;
notebook 04 reads those scores and makes the tables and figures.

## What changed from v1

| | v1 | v2 |
|---|---|---|
| Data | Wednesday only | All five days (2.57 million flows, 14 attack types) |
| What the tree learns | The name of each attack tool | **Attack or normal** |
| Features | Lists A-D, with and without shortcut features | Lists A-D **without** the shortcut features only |
| Test set | Random 20% of the same day | Four tests (below), three of them on traffic the tree has not seen |
| Choosing the tree size | Validation set cut at random | Validation set that comes **later in time** than the training set |

## The four tests

| Test | Train on | Test on | Question it answers |
|---|---|---|---|
| **1. Random split** | random 60% of all flows | random 20% | v1's method, kept only as the comparison |
| **2. Later traffic** | first 60% of every attack and of normal traffic | last 20% | Does it still work later the same day? |
| **3. New attack** | everything in the training part *except one attack type* | that attack, plus later normal traffic | Does it catch an attack tool it has never seen? (14 runs, one per attack) |
| **4. New day** | training part of four days | the whole fifth day | Does it work on a day it has never seen? (5 runs, one per day) |

In every test the validation set is the middle 20% of the training days (or a random 20% in
test 1), and is used only to choose the tree size.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))  # finds helpers.py

import os
import time
import numpy as np
import pandas as pd
import sklearn
from joblib import Parallel, delayed
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

import helpers as h

QUICK = os.environ.get("QUICK", "0") == "1"       # True = small, fast run to check the code
OUT_DIR = h.RESULTS_DIR.parent / "results_quick_test" if QUICK else h.RESULTS_DIR
OUT_DIR.mkdir(exist_ok=True)
print("scikit-learn", sklearn.__version__, "| QUICK =", QUICK)

scikit-learn 1.9.1 | QUICK = False


## Load the data

In [2]:
data = h.load_all_days()
if QUICK:
    data = data.iloc[::25].reset_index(drop=True)         # every 25th flow, order kept
    data["part"] = h.time_parts(data)

labels = data["Label"].to_numpy()
day = data["day"].to_numpy()
part = data["part"].to_numpy()
is_attack = (labels != h.BENIGN).astype(np.int8)          # what the trees learn: 1 = attack, 0 = normal
ATTACKS = sorted(set(labels) - {h.BENIGN})

LISTS = h.load_feature_lists()
if QUICK:
    LISTS = {k: LISTS[k] for k in ["C-NS", "D-NS"]}
X_BY_LIST = {name: data[cols].to_numpy(dtype=np.float32) for name, cols in LISTS.items()}
del data

print(f"{len(labels):,} flows, {is_attack.sum():,} attacks of {len(ATTACKS)} types")
print({k: len(v) for k, v in LISTS.items()})

2,572,640 flows, 425,741 attacks of 14 types
{'A-NS': 55, 'B-NS': 36, 'C-NS': 13, 'D-NS': 4}


## Two checks before the long run

To save time, each tree is grown **once** at full size and then read at every smaller size:

- **Depth limit:** the full tree is cut off at depth *d*. A tree is grown from the top down,
  one best question at a time, so the top *d* levels of the full tree are themselves a tree
  that `max_depth=d` could have grown. The only difference from re-training is how ties
  between two equally good questions are broken, which scikit-learn does at random. The
  check below measures how often the two disagree.
- **Pruning:** pruning the full tree should give the same tree as training a new one with
  `ccp_alpha` set.

Both are checked here on real data before being relied on.

In [3]:
check_list = "C-NS"
Xc = X_BY_LIST[check_list]
tr, te = part == "train", part == "test"
full = h.grow_tree(Xc[tr], is_attack[tr])
cut = h.predict_at_depths(full, Xc[te])
depth_of_node = h.node_depths(full)

for d in [1, 3, 6, 10, 15]:
    retrained = DecisionTreeClassifier(random_state=h.SEED, max_depth=d).fit(Xc[tr], is_attack[tr])
    agree = (cut[d][0] == retrained.predict(Xc[te])).mean()
    assert agree > 0.995
    print(f"depth {d:2d}: cut tree and re-trained tree agree on {agree:.3%} of test flows "
          f"({h.nodes_at_depth(depth_of_node, d)} and {retrained.tree_.node_count} nodes)")
assert (cut[None][0] == full.predict(Xc[te])).all()

for strength in [1e-6, 1e-4]:
    retrained = DecisionTreeClassifier(random_state=h.SEED, ccp_alpha=strength).fit(Xc[tr], is_attack[tr])
    pruned = h.prune(full, strength)
    assert pruned.tree_.node_count == retrained.tree_.node_count
    assert (pruned.predict(Xc[te]) == retrained.predict(Xc[te])).all()
    print(f"pruning {strength:g}: pruned tree and re-trained tree are identical, {pruned.tree_.node_count} nodes each")

depth  1: cut tree and re-trained tree agree on 100.000% of test flows (3 and 3 nodes)


depth  3: cut tree and re-trained tree agree on 100.000% of test flows (15 and 15 nodes)


depth  6: cut tree and re-trained tree agree on 100.000% of test flows (95 and 95 nodes)


depth 10: cut tree and re-trained tree agree on 99.998% of test flows (443 and 443 nodes)


depth 15: cut tree and re-trained tree agree on 99.949% of test flows (1321 and 1319 nodes)


pruning 1e-06: pruned tree and re-trained tree are identical, 1475 nodes each


pruning 0.0001: pruned tree and re-trained tree are identical, 107 nodes each


## Define the four tests

Each run is described by three sets of rows (train, validation, test) and the list of
attack types that are kept out of training.

In [4]:
runs = []

# Test 1: random split (v1's method), 60/20/20, same class mix in each part
idx = np.arange(len(labels))
rest, test_idx = train_test_split(idx, test_size=0.20, random_state=h.SEED, stratify=labels)
train_idx, val_idx = train_test_split(rest, test_size=0.25, random_state=h.SEED, stratify=labels[rest])
mask = lambda rows: np.isin(idx, rows)
runs.append({"test": "1 random split", "held_out": "-", "unseen": [],
             "train": mask(train_idx), "validation": mask(val_idx), "test_rows": mask(test_idx)})

# Test 2: later traffic
runs.append({"test": "2 later traffic", "held_out": "-", "unseen": [],
             "train": part == "train", "validation": part == "validation", "test_rows": part == "test"})

# Test 3: new attack (one run per attack type)
for attack in ATTACKS:
    this = labels == attack
    runs.append({"test": "3 new attack", "held_out": attack, "unseen": [attack],
                 "train": (part == "train") & ~this, "validation": (part == "validation") & ~this,
                 "test_rows": ((part == "test") & ~this) | this})   # all flows of the unseen attack

# Test 4: new day (one run per day)
for d in h.DAYS:
    this = day == d
    runs.append({"test": "4 new day", "held_out": d, "unseen": sorted(set(labels[this]) - {h.BENIGN}),
                 "train": (part == "train") & ~this, "validation": (part == "validation") & ~this,
                 "test_rows": this})

if QUICK:
    runs = [r for r in runs if r["held_out"] in ("-", "DoS Hulk", "Friday")]
for r in runs:
    assert not (r["train"] & r["test_rows"]).any() and not (r["train"] & r["validation"]).any()
    assert not np.isin(labels[r["train"]], r["unseen"]).any()       # unseen attacks really are unseen
pd.DataFrame([{"test": r["test"], "held out": r["held_out"], "train flows": r["train"].sum(),
               "validation flows": r["validation"].sum(), "test flows": r["test_rows"].sum()} for r in runs])

,test,held out,train flows,validation flows,test flows
0,1 random split,-,1543584,514528,514528
1,2 later traffic,-,1543592,514528,514520
2,3 new attack,Bot,1542423,514138,516079
3,3 new attack,DDoS,1466783,488925,616932
4,3 new attack,DoS GoldenEye,1537420,512471,522749
5,3 new attack,DoS Hulk,1439884,479959,652797
6,3 new attack,DoS Slowhttptest,1540455,513482,518703
7,3 new attack,DoS slowloris,1540361,513451,518828
8,3 new attack,FTP-Patator,1540033,513342,519265
9,3 new attack,Heartbleed,1543585,514526,514529


## Run everything

For each run and each feature list: grow one full tree on the training rows, then score it
on the validation rows and the test rows at

- 12 **depth limits** (1 to 20, and no limit), and
- 12 **pruning strengths**.

Recorded for every size: nodes, compact size, average comparisons per flow, false-alarm
rate, and recall for attacks the tree saw in training and for attacks it did not, plus the
recall of every single attack type on both the validation and the test rows. Results are
saved after every run, so an interrupted run loses little.

In [5]:
def describe(run, list_name, method, setting, tree, n_nodes, depth_reached):
    first = tree.tree_
    return {"test": run["test"], "held_out": run["held_out"], "feature_list": list_name,
            "features": len(LISTS[list_name]), "method": method, "setting": setting,
            "nodes": n_nodes, "depth_reached": depth_reached, "compact_kb": round(h.compact_kb(n_nodes), 3),
            "first_question": f"{LISTS[list_name][first.feature[0]]} <= {first.threshold[0]:g}"}


def record(base, said_val, said_test, comparisons, run, tree_rows, attack_rows):
    """Score one tree size on the validation rows and the test rows and store the results."""
    val_summary, val_per_attack = h.score(said_val, labels[run["validation"]])
    test_summary, test_per_attack = h.score(said_test, labels[run["test_rows"]], run["unseen"])
    tree_rows.append({**base, "avg_comparisons": round(comparisons, 3), **test_summary,
                      "val_false_alarm_rate": val_summary["false_alarm_rate"],
                      "val_attack_recall": val_summary["seen_attack_recall"]})
    key = {k: base[k] for k in ("test", "held_out", "feature_list", "method", "setting")}
    attack_rows.extend({**key, "scored_on": "validation", **a} for a in val_per_attack)
    attack_rows.extend({**key, "scored_on": "test", **a} for a in test_per_attack)


tree_rows, attack_rows = [], []
started = time.time()
for n, run in enumerate(runs, 1):
    # Grow the full trees of this run, two at a time (one per CPU core)
    y_train = is_attack[run["train"]]
    grown = Parallel(n_jobs=2, prefer="threads")(
        delayed(h.grow_tree)(X[run["train"]], y_train) for X in X_BY_LIST.values())

    for (list_name, X), tree in zip(X_BY_LIST.items(), grown):
        X_val, X_test = X[run["validation"]], X[run["test_rows"]]
        depth_of_node = h.node_depths(tree)

        # --- depth limits: read the same tree at every depth ---
        at_val, at_test = h.predict_at_depths(tree, X_val), h.predict_at_depths(tree, X_test)
        for d in h.DEPTHS:
            n_nodes = h.nodes_at_depth(depth_of_node, d)
            reached = int(depth_of_node.max()) if d is None else min(d, int(depth_of_node.max()))
            base = describe(run, list_name, "depth limit", "none" if d is None else str(d), tree, n_nodes, reached)
            record(base, at_val[d][0] == 1, at_test[d][0] == 1, at_test[d][1], run, tree_rows, attack_rows)

        # --- pruning ---
        for strength in h.PRUNING_STRENGTHS:
            pruned = h.prune(tree, strength)
            pruned_depths = h.node_depths(pruned)
            leaf = pruned.apply(X_test)
            base = describe(run, list_name, "pruning", f"{strength:g}", pruned, pruned.tree_.node_count, int(pruned_depths.max()))
            record(base, pruned.predict(X_val) == 1, pruned.predict(X_test) == 1,
                   float(pruned_depths[leaf].mean()), run, tree_rows, attack_rows)

    pd.DataFrame(tree_rows).to_csv(OUT_DIR / "03_all_trees.csv", index=False)
    pd.DataFrame(attack_rows).to_csv(OUT_DIR / "03_recall_per_attack.csv", index=False)
    print(f"[{n:2d}/{len(runs)}] {run['test']:16s} {run['held_out']:26s} done, {(time.time() - started) / 60:5.1f} min so far", flush=True)

trees = pd.DataFrame(tree_rows)
print(f"\n{len(trees):,} rows saved to 03_all_trees.csv, {len(attack_rows):,} rows to 03_recall_per_attack.csv")

[ 1/21] 1 random split   -                          done,   1.6 min so far


[ 2/21] 2 later traffic  -                          done,   3.2 min so far


[ 3/21] 3 new attack     Bot                        done,   5.0 min so far


[ 4/21] 3 new attack     DDoS                       done,   6.8 min so far


[ 5/21] 3 new attack     DoS GoldenEye              done,   8.3 min so far


[ 6/21] 3 new attack     DoS Hulk                   done,   9.7 min so far


... (12 outputs left out)

[19/21] 4 new day        Wednesday                  done,  27.4 min so far


[20/21] 4 new day        Thursday                   done,  28.5 min so far


[21/21] 4 new day        Friday                     done,  29.7 min so far



2,016 rows saved to 03_all_trees.csv, 48,384 rows to 03_recall_per_attack.csv


## First look: the full-size trees

One line per test and feature list, averaged over the runs of that test. The detailed
analysis is in notebook 04.

In [6]:
full = trees[(trees["method"] == "depth limit") & (trees["setting"] == "none")]
look = (full.groupby(["test", "feature_list"], sort=False)
        [["nodes", "false_alarm_rate", "seen_attack_recall", "unseen_attack_recall"]].mean())
look[["false_alarm_rate", "seen_attack_recall", "unseen_attack_recall"]] *= 100
look.round(2)

nodes  false_alarm_rate  seen_attack_recall  \
test            feature_list                                                   
1 random split  A-NS           5493.00              0.11               99.17   
                B-NS           6659.00              0.14               99.07   
                C-NS           8303.00              0.16               98.93   
                D-NS          24277.00              0.49               97.36   
2 later traffic A-NS           4043.00              0.08               94.69   
                B-NS           5045.00              0.09               94.91   
                C-NS           5699.00              0.16               93.41   
                D-NS          24897.00              0.24               90.15   
3 new attack    A-NS           3826.71              0.07               94.91   
                B-NS           4765.71              0.09               94.55   
                C-NS           5343.29              0.14               93.76   
                D-NS          23202.43              0.22               89.46   
4 new day       A-NS           3218.60              1.18                 NaN   
                B-NS           3913.80              1.77                 NaN   
                C-NS           4293.80              0.21                 NaN   
                D-NS          19428.20              0.88                 NaN   

                              unseen_attack_recall  
test            feature_list                        
1 random split  A-NS                           NaN  
                B-NS                           NaN  
                C-NS                           NaN  
                D-NS                           NaN  
2 later traffic A-NS                           NaN  
                B-NS                           NaN  
                C-NS                           NaN  
                D-NS                           NaN  
3 new attack    A-NS                         32.32  
                B-NS                         28.05  
                C-NS                         27.84  
                D-NS                         11.08  
4 new day       A-NS                         24.23  
                B-NS                          4.44  
                C-NS                          6.69  
                D-NS                         15.11